# ⚖️ Weighted Averaging — K = 1 to 5

This notebook supports **“DCMC-Fusion: Data-Centric and Model-Centric Ensemble Learning for Audio Deepfake Detection”**, by **Dora Ballesteros and Carlos Guerra**.

**Purpose:** rank the four spectral representations within each of five architectures using **training-only macro F1**, assign ranks **4, 3, 2, 1**, average these ranks across architectures to obtain **four global representation weights**, and evaluate weighted probability averaging on the **same held-out test samples** for all 31 architecture combinations (K=1,…,5).

### 📥 Inputs
1. `inference_probabilities.csv` — precomputed detector scores (0–100) and `label` (0 natural, 1 synthetic).
2. `experiment_split.json` — the shared train/test indices and sample-order metadata.

### 📤 Outputs
- `weighted_averaging_individual_train_metrics.csv` — training-only F1 for all 20 detectors.
- `weighted_averaging_architecture_ranks.csv` — within-architecture ranks (4 best, 1 worst).
- `weighted_averaging_representation_weights.csv` — averaged and normalized global weights.
- `weighted_averaging_global_metrics.csv` — test metrics for all 31 combinations.
- `weighted_averaging_class_metrics.csv` — test metrics by class for all combinations.

**Methodological safeguards:** No test labels are used to determine weights. Ties in training F1 receive average ranks; scores are thresholded at 50. This is a **rank-weighted averaging** baseline, not a test-optimized weighting method.

In [1]:
from pathlib import Path
from itertools import combinations
import json
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

# Input and output directory
DATA_DIR = Path(".")
OUTPUT_DIR = DATA_DIR

# Architectures and spectral representations
ARCHITECTURES = ['resnet50', 'densenet121', 'inception_v3', 'resnet34', 'vgg16']
REPRESENTATIONS = ['mel', 'log', 'dwt', 'cqt']
THRESHOLD = 50.0

# Load input files
inference = pd.read_csv(DATA_DIR / 'inference_probabilities.csv')

with open(DATA_DIR / 'experiment_split.json', encoding='utf-8') as f:
    split = json.load(f)

# Validate inference dataset
expected_columns = [
    f'{a}_{r}'
    for a in ARCHITECTURES
    for r in REPRESENTATIONS
]

missing = set(['audio', 'label', *expected_columns]) - set(inference.columns)

if missing:
    raise ValueError(f'Missing inference columns: {sorted(missing)}')

if inference['label'].isna().any() or not inference['label'].isin([0, 1]).all():
    raise ValueError('Labels must be binary: 0 or 1.')

if inference[expected_columns].isna().any().any():
    raise ValueError('Inference scores contain missing values.')

if not np.isfinite(inference[expected_columns].to_numpy(dtype=float)).all():
    raise ValueError('Inference scores contain non-finite values.')

if not inference[expected_columns].stack().between(0, 100).all():
    raise ValueError('Inference scores must be between 0 and 100.')

# Verify consistency with the original experimental split
if list(inference['audio'].astype(str)) != list(map(str, split['audio_order'])):
    raise ValueError('Audio order does not match experiment_split.json.')

if list(inference['label'].astype(int)) != list(map(int, split['label_order'])):
    raise ValueError('Label order does not match experiment_split.json.')

if expected_columns != list(split['feature_columns']):
    raise ValueError('Feature order does not match experiment_split.json.')

# Recover training and test partitions
train_idx = np.asarray(split['train_indices'], dtype=int)
test_idx = np.asarray(split['test_indices'], dtype=int)

all_idx = np.concatenate([train_idx, test_idx])

if (
    len(all_idx) != len(inference)
    or len(np.unique(all_idx)) != len(inference)
    or not np.array_equal(np.sort(all_idx), np.arange(len(inference)))
):
    raise ValueError(
        'Train/test indices must be valid, disjoint, and cover all samples exactly once.'
    )

train = inference.iloc[train_idx]
test = inference.iloc[test_idx]

print(f'Loaded {len(inference)} samples: {len(train)} train / {len(test)} test.')

Loaded 2484 samples: 1738 train / 746 test.


## 1. Rank spectral representations within each architecture

For each architecture, compute the macro F1 of its four detectors **on training samples only**. Convert scores to predictions with threshold 50. Assign rank weights **4 (best) → 1 (worst)**; equal F1 values receive average ranks.

In [2]:
# Evaluate individual detectors using training data only
individual_rows = []

for arch in ARCHITECTURES:
    for rep in REPRESENTATIONS:

        column = f'{arch}_{rep}'

        # Convert inference scores into binary predictions
        y_pred = (
            train[column].to_numpy(dtype=float) >= THRESHOLD
        ).astype(int)

        # Calculate macro F1 for each architecture-representation pair
        individual_rows.append({
            'architecture': arch,
            'representation': rep,
            'model': column,
            'train_f1_macro': f1_score(
                train['label'],
                y_pred,
                average='macro',
                zero_division=0
            )
        })

individual = pd.DataFrame(individual_rows)

# Assign rank-based weights within each architecture
# Worst representation = 1; best representation = 4
individual['rank_weight'] = (
    individual.groupby('architecture')['train_f1_macro']
    .transform(
        lambda scores: scores.rank(
            method='average',
            ascending=True
        )
    )
)

# Save individual performance and assigned weights
individual.to_csv(
    OUTPUT_DIR / 'weighted_averaging_individual_train_metrics.csv',
    index=False
)

# Create architecture-representation weight matrix
rank_table = individual.pivot(
    index='architecture',
    columns='representation',
    values='rank_weight'
)

rank_table = rank_table.reindex(
    index=ARCHITECTURES,
    columns=REPRESENTATIONS
)

rank_table.to_csv(
    OUTPUT_DIR / 'weighted_averaging_architecture_ranks.csv'
)

# Display results
print('Rank weights by architecture (4 = best, 1 = worst):')
display(rank_table)

print('Training F1-macro of individual detectors:')
display(
    individual.sort_values(
        ['architecture', 'train_f1_macro'],
        ascending=[True, False]
    )
)

Rank weights by architecture (4 = best, 1 = worst):


representation,mel,log,dwt,cqt
architecture,,,,
resnet50,3.0,4.0,1.0,2.0
densenet121,1.0,4.0,2.0,3.0
inception_v3,3.0,4.0,2.0,1.0
resnet34,4.0,3.0,2.0,1.0
vgg16,3.0,4.0,2.0,1.0


Training F1-macro of individual detectors:


,architecture,representation,model,train_f1_macro,rank_weight
5,densenet121,log,densenet121_log,0.653569,4.0
7,densenet121,cqt,densenet121_cqt,0.555119,3.0
6,densenet121,dwt,densenet121_dwt,0.427317,2.0
4,densenet121,mel,densenet121_mel,0.357081,1.0
9,inception_v3,log,inception_v3_log,0.639563,4.0
8,inception_v3,mel,inception_v3_mel,0.618776,3.0
10,inception_v3,dwt,inception_v3_dwt,0.516835,2.0
11,inception_v3,cqt,inception_v3_cqt,0.348078,1.0
12,resnet34,mel,resnet34_mel,0.473974,4.0
13,resnet34,log,resnet34_log,0.461571,3.0




## 2. Compute average rank-based weights and normalize global representation weights

Let $w_{a,s}$ denote the rank-based weight assigned to representation $s$ within architecture $a$. For each representation, the rank-based weights are averaged across the five architectures and subsequently normalized as follows:

$$
\bar{w}_s = \frac{1}{5}\sum_{a=1}^{5}w_{a,s},
\qquad
\alpha_s = \frac{\bar{w}_s}{\sum_{t=1}^{4}\bar{w}_t}.
$$

Since each architecture distributes a total of 10 rank points across the four representations, the averaged weights also sum to 10 before normalization. Consequently, the four normalized global weights satisfy $\sum_{s=1}^{4}\alpha_s=1$ and remain fixed across all values of $K$.




In [3]:
# Calculate the mean rank weight for each spectral representation
weights = pd.DataFrame({
    'representation': REPRESENTATIONS,
    'mean_rank_weight': [
        rank_table[r].mean() for r in REPRESENTATIONS
    ]
})

# Normalize weights so that their sum equals 1
weights['normalized_weight'] = (
    weights['mean_rank_weight'] /
    weights['mean_rank_weight'].sum()
)

# Verify weight normalization
assert np.isclose(weights['normalized_weight'].sum(), 1.0)

# Save global representation weights
weights.to_csv(
    OUTPUT_DIR / 'weighted_averaging_representation_weights.csv',
    index=False
)

# Create a dictionary for subsequent weighted averaging
weight_by_rep = dict(zip(
    weights['representation'],
    weights['normalized_weight']
))

# Display results
print('Global representation weights:')
display(weights)

Global representation weights:


,representation,mean_rank_weight,normalized_weight
0,mel,2.8,0.28
1,log,3.8,0.38
2,dwt,1.8,0.18
3,cqt,1.6,0.16



## 3. Evaluate all architecture combinations on the held-out test partition

For a combination $C$ of $K$ architectures, the weighted synthetic score is computed as follows:

$$
P_{\mathrm{weighted}}(x;C) =
\frac{1}{K}
\sum_{a \in C}
\sum_{s \in \{\mathrm{mel,log,dwt,cqt}\}}
\alpha_s P_{a,s}(x).
$$

Here, $\alpha_s$ denotes the normalized global weight assigned to representation $s$, and $P_{a,s}(x)$ is the predicted synthetic-class score (0–100) for sample $x$, obtained from architecture $a$ using representation $s$.

A sample is classified as **synthetic (`1`)** when $P_{\mathrm{weighted}}(x;C) \geq 50$; otherwise, it is classified as **real (`0`)**.

For each value of $K$, all $\binom{5}{K}$ possible architecture combinations are evaluated, resulting in **31 configurations** across $K=1,\ldots,5$.

Performance metrics are computed using the **same held-out test partition** employed for DCMC-Fusion and the voting-based baselines, ensuring a consistent evaluation across fusion methods.



In [4]:
# Initialize result containers
global_rows = []
class_rows = []

# Ground-truth labels from the held-out test partition
y_true = test['label'].to_numpy(dtype=int)

# Evaluate all ensemble sizes (K = 1, ..., 5)
for K in range(1, len(ARCHITECTURES) + 1):

    # Generate all combinations of K architectures
    for combo in combinations(ARCHITECTURES, K):

        # Initialize weighted scores for all test samples
        score = np.zeros(len(test), dtype=float)

        # Apply global representation weights
        for arch in combo:
            for rep in REPRESENTATIONS:

                score += (
                    weight_by_rep[rep] / K
                ) * test[f'{arch}_{rep}'].to_numpy(dtype=float)

        # Verify that weighted scores remain within the expected range
        assert np.all((score >= -1e-10) & (score <= 100 + 1e-10))

        # Convert weighted scores into binary predictions
        y_pred = (score >= THRESHOLD).astype(int)

        # Configuration information
        common = {
            'K': K,
            'architectures': '+'.join(combo),
            'n_features': 4 * K,
            'method': 'Rank-Weighted Averaging'
        }

        # Calculate global performance metrics
        global_rows.append({
            **common,
            'accuracy': accuracy_score(y_true, y_pred),
            'f1_macro': f1_score(
                y_true, y_pred,
                average='macro',
                zero_division=0
            ),
            'precision_macro': precision_score(
                y_true, y_pred,
                average='macro',
                zero_division=0
            ),
            'recall_macro': recall_score(
                y_true, y_pred,
                average='macro',
                zero_division=0
            )
        })

        # Calculate class-specific performance metrics
        for cls, name in [(0, 'natural'), (1, 'synthetic')]:

            class_rows.append({
                **common,
                'class_label': cls,
                'class_name': name,
                'support': int(np.sum(y_true == cls)),
                'precision': precision_score(
                    y_true, y_pred,
                    labels=[cls],
                    average='macro',
                    zero_division=0
                ),
                'recall': recall_score(
                    y_true, y_pred,
                    labels=[cls],
                    average='macro',
                    zero_division=0
                ),
                'f1': f1_score(
                    y_true, y_pred,
                    labels=[cls],
                    average='macro',
                    zero_division=0
                )
            })

# Convert results into DataFrames
global_metrics = pd.DataFrame(global_rows)
class_metrics = pd.DataFrame(class_rows)

# Verify the expected number of configurations
assert len(global_metrics) == 31
assert len(class_metrics) == 62
assert global_metrics.groupby('K').size().tolist() == [5, 10, 10, 5, 1]

# Save evaluation results
global_metrics.to_csv(
    OUTPUT_DIR / 'weighted_averaging_global_metrics.csv',
    index=False
)

class_metrics.to_csv(
    OUTPUT_DIR / 'weighted_averaging_class_metrics.csv',
    index=False
)

# Identify the best configuration for each ensemble size
best_per_k = (
    global_metrics
    .sort_values('f1_macro', ascending=False)
    .drop_duplicates('K')
    .sort_values('K')
    .reset_index(drop=True)
)

print('Best rank-weighted averaging combination per K:')

display(
    best_per_k[
        ['K', 'architectures', 'accuracy',
         'f1_macro', 'precision_macro', 'recall_macro']
    ]
)

# Analyze performance distribution by ensemble size
print('Distribution of macro F1 by K:')

display(
    global_metrics.groupby('K')['f1_macro']
    .agg(['count', 'mean', 'std', 'min', 'max'])
)

print('CSV files saved in:', OUTPUT_DIR.resolve())

Best rank-weighted averaging combination per K:


,K,architectures,accuracy,f1_macro,precision_macro,recall_macro
0,1,inception_v3,0.619303,0.610200,0.631595,0.619303
1,2,inception_v3+vgg16,0.604558,0.596251,0.613933,0.604558
2,3,resnet50+inception_v3+vgg16,0.591153,0.581552,0.600364,0.591153
3,4,resnet50+densenet121+inception_v3+vgg16,0.577748,0.563886,0.589073,0.577748
4,5,resnet50+densenet121+inception_v3+resnet34+vgg16,0.565684,0.515542,0.612089,0.565684


Distribution of macro F1 by K:


,count,mean,std,min,max
K,,,,,
1,5,0.541693,0.084437,0.404707,0.610200
2,10,0.527902,0.061605,0.430660,0.596251
3,10,0.523171,0.043397,0.465335,0.581552
4,5,0.526706,0.023209,0.504672,0.563886
5,1,0.515542,NaN,0.515542,0.515542


CSV files saved in: /content
